# Synthetic Hierarchical Time Series Data Generation

Generates tables for testing **hierarchical reconciliation** with the MMF pipeline.

Produces leaf-level series with separate hierarchy columns, then calls `run_aggregation()` to build all hierarchy levels and save the summing matrix and level tags.

**Output tables:**
- `{catalog}.{schema}.{use_case}_train_data` — all hierarchy levels (`unique_id` encodes level, e.g. `USA/California/Store1`)
- `{catalog}.{schema}.{use_case}_hierarchy_S` — summing matrix (S_df)
- `{catalog}.{schema}.{use_case}_hierarchy_tags` — level tags

**Hierarchy structure:**
```
Total
├── USA
│   ├── USA/California
│   │   ├── USA/California/Store1
│   │   └── USA/California/Store2
│   └── USA/NewYork
│       └── USA/NewYork/Store3
└── Europe
    ├── Europe/France
    │   └── Europe/France/Store4
    └── Europe/Germany
        └── Europe/Germany/Store5
```

## 1. Parameters

In [ ]:
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "catalog_timeseries"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "synthetic"

try:
    use_case = dbutils.widgets.get("use_case")
except Exception:
    use_case = "mmf_e2e"

try:
    n_months = int(dbutils.widgets.get("n_months"))
except Exception:
    n_months = 36

train_table = f"{catalog}.{schema}.{use_case}_train_data"
print(f"Train table:    {train_table}")
print(f"Hierarchy S:    {catalog}.{schema}.{use_case}_hierarchy_S")
print(f"Hierarchy tags: {catalog}.{schema}.{use_case}_hierarchy_tags")
print(f"Months:         {n_months}")

## 2. Setup

In [ ]:
import subprocess, sys

_current_user = spark.sql("SELECT current_user() AS u").collect()[0]["u"]
_workspace_path = f"/Workspace/Repos/{_current_user}/many-model-forecasting"
subprocess.check_call([sys.executable, "-m", "pip", "install", f"{_workspace_path}[hierarchical]", "--quiet"])
dbutils.library.restartPython()

In [ ]:
import sys
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

_current_user = spark.sql("SELECT current_user() AS u").collect()[0]["u"]
_workspace_path = f"/Workspace/Repos/{_current_user}/many-model-forecasting"
sys.path.insert(0, _workspace_path)
from mmf_sa import run_aggregation

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

## 3. Generate Leaf-Level Series

In [ ]:
np.random.seed(42)

# 5 leaf series with hierarchy columns
LEAVES = [
    {"unique_id": "Store1", "country": "USA",    "region": "California", "store": "Store1"},
    {"unique_id": "Store2", "country": "USA",    "region": "California", "store": "Store2"},
    {"unique_id": "Store3", "country": "USA",    "region": "NewYork",    "store": "Store3"},
    {"unique_id": "Store4", "country": "Europe", "region": "France",     "store": "Store4"},
    {"unique_id": "Store5", "country": "Europe", "region": "Germany",    "store": "Store5"},
]

# Base levels and seasonal amplitudes per store
BASE   = {"Store1": 200, "Store2": 150, "Store3": 180, "Store4": 130, "Store5": 160}
SLOPE  = {"Store1": 1.2, "Store2": 0.8, "Store3": 1.0, "Store4": 0.6, "Store5": 0.9}
AMP    = {"Store1":  20, "Store2":  15, "Store3":  18, "Store4":  12, "Store5":  14}
NOISE  = {"Store1":   8, "Store2":   6, "Store3":   7, "Store4":   5, "Store5":   6}

end_date = pd.Timestamp.today().replace(day=1)
dates = pd.date_range(end=end_date, periods=n_months, freq="MS")

rows = []
for leaf in LEAVES:
    sid = leaf["store"]
    for t, ds in enumerate(dates):
        y = (
            BASE[sid]
            + SLOPE[sid] * t
            + AMP[sid] * np.sin(2 * np.pi * t / 12)
            + np.random.normal(0, NOISE[sid])
        )
        rows.append({
            "unique_id": sid,
            "ds":        ds.date(),
            "y":         max(0.0, round(y, 2)),
            "country":   leaf["country"],
            "region":    leaf["region"],
            "store":     leaf["store"],
        })

leaf_pdf = pd.DataFrame(rows)
leaf_sdf = spark.createDataFrame(leaf_pdf)

print(f"Leaf series generated: {leaf_pdf['unique_id'].nunique()} series × {n_months} months = {len(leaf_pdf):,} rows")
print(f"Date range: {leaf_pdf['ds'].min()} → {leaf_pdf['ds'].max()}")
display(leaf_pdf.groupby("unique_id")[["y"]].agg(["mean","min","max"]).round(1))

## 4. Aggregate and Save All Hierarchy Levels

In [ ]:
# Write leaf data to Delta first (run_aggregation reads from Delta)
(
    leaf_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(train_table)
)
print(f"Leaf data written to: {train_table}")

# run_aggregation() reads leaf data, builds all hierarchy levels,
# overwrites train_table with all levels, and saves _hierarchy_S and _hierarchy_tags
run_aggregation(
    spark=spark,
    train_table=train_table,
    hierarchy_cols=["country", "region", "store"],
    hierarchy_s_table=f"{catalog}.{schema}.{use_case}_hierarchy_S",
    hierarchy_tags_table=f"{catalog}.{schema}.{use_case}_hierarchy_tags",
)
print("✓ Aggregation complete — all hierarchy levels written")

## 5. Verify

In [ ]:
print("=== Train table (all hierarchy levels) ===")
display(spark.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT unique_id)         AS n_series,
        MIN(ds)                           AS min_date,
        MAX(ds)                           AS max_date,
        ROUND(AVG(y), 2)                  AS avg_y
    FROM {train_table}
""").toPandas())

print("\nSeries by hierarchy level (classified by number of '/' in unique_id):")
display(spark.sql(f"""
    SELECT
        CASE SIZE(SPLIT(unique_id, '/')) - 1
            WHEN 0 THEN 'country (level 1)'
            WHEN 1 THEN 'country/region (level 2)'
            WHEN 2 THEN 'country/region/store — leaf (level 3)'
            ELSE CONCAT(CAST(SIZE(SPLIT(unique_id, '/')) - 1 AS STRING), ' slashes')
        END AS level,
        COUNT(DISTINCT unique_id) AS n_series
    FROM {train_table}
    GROUP BY 1 ORDER BY 1
""").toPandas())

print("\n=== Hierarchy S matrix (first 5 rows) ===")
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.{use_case}_hierarchy_S LIMIT 5").toPandas())

print("\n=== Hierarchy tags ===")
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.{use_case}_hierarchy_tags").toPandas())